In [1]:
import os
import glob
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Flatten, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings('ignore')
print("imported all")

imported all


In [2]:
# --- Load Parquet Files ---
def load_parquet_files(data_dir, files_to_load=8):
    parquet_files = sorted(glob.glob(os.path.join(data_dir, "*.parquet")))
    assert len(parquet_files) >= files_to_load, "Not enough .parquet files found."
    data_chunks = []
    for file in parquet_files[:files_to_load]:
        print(f"Loading: {os.path.basename(file)}")
        pf = pd.read_parquet(file, engine="pyarrow")
        for col in pf.select_dtypes('number').columns:
            if 'int' in str(pf[col].dtype):
                pf[col] = pd.to_numeric(pf[col], downcast='integer')
            elif 'float' in str(pf[col].dtype):
                pf[col] = pd.to_numeric(pf[col], downcast='float')
        for col in pf.select_dtypes('object').columns:
            if pf[col].nunique() < 0.5 * len(pf):
                pf[col] = pf[col].astype('category')
        data_chunks.append(pf)
        gc.collect()
    df = pd.concat(data_chunks, ignore_index=True)
    return df

DATA_DIR = r"C:\Users\MY PC\OneDrive\Desktop\self-healing-cybersecurity-framework\data"

print("\n# Loading data efficiently from Parquet files...")
df = load_parquet_files(DATA_DIR, files_to_load=8)
print(f"Loaded dataframe shape: {df.shape}")



# Loading data efficiently from Parquet files...
Loading: Benign-Monday-no-metadata.parquet
Loading: Botnet-Friday-no-metadata.parquet
Loading: Bruteforce-Tuesday-no-metadata.parquet
Loading: DDoS-Friday-no-metadata.parquet
Loading: DoS-Wednesday-no-metadata.parquet
Loading: Infiltration-Thursday-no-metadata.parquet
Loading: NUSW-NB15_GT.CSV.parquet
Loading: Portscan-Friday-no-metadata.parquet
Loaded dataframe shape: (2332337, 89)


In [3]:
# --- Data Cleaning ---
drop_cols = [
    'Flow ID', 'Src IP', 'Dst IP', 'Dst Port', 'Timestamp', 'Source IP',
    'Destination IP', 'Fwd Header Length.1', 'Start time', 'Last time',
    'Attack Name', 'Attack Reference', '.', 'Attack category', 'Attack subcategory',
    'Source Port', 'Destination Port'
]

df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True, errors='ignore')

if 'Label' in df.columns:
    df['Label'] = df['Label'].astype(str).str.strip().str.upper()
else:
    df['Label'] = 'BENIGN'  # Default to benign if no Label column

print("\nLabel distribution after cleaning:")
print(df['Label'].value_counts())



Label distribution after cleaning:
Label
BENIGN              1823641
NAN                  174347
DOS HULK             172846
DDOS                 128014
DOS GOLDENEYE         10286
FTP-PATATOR            5931
DOS SLOWLORIS          5385
DOS SLOWHTTPTEST       5228
SSH-PATATOR            3219
PORTSCAN               1956
BOT                    1437
INFILTRATION             36
HEARTBLEED               11
Name: count, dtype: int64


In [4]:
# --- Split Data for Supervised and Unsupervised ---
benign_data = df[df['Label'] == 'BENIGN'].copy()
attack_data = df[df['Label'] != 'BENIGN'].copy()

supervised_df = pd.concat([benign_data.sample(frac=0.3, random_state=42), attack_data], axis=0)
unsupervised_df = benign_data.drop(supervised_df.index, errors='ignore')

y_supervised = supervised_df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1).values
X_supervised = supervised_df.drop(columns=['Label'])
X_unsupervised = unsupervised_df.drop(columns=['Label'])

# Convert object columns to numeric
for df_part in [X_supervised, X_unsupervised]:
    for col in df_part.columns:
        if df_part[col].dtype == 'object':
            df_part[col] = pd.to_numeric(df_part[col], errors='coerce').fillna(0)

# Remove constant columns
const_cols = X_supervised.columns[X_supervised.nunique() <= 1]
X_supervised.drop(columns=const_cols, inplace=True)
X_unsupervised.drop(columns=const_cols, inplace=True)


In [5]:
X_supervised.fillna(0, inplace=True)
X_unsupervised.fillna(0, inplace=True)


In [6]:
# --- Feature Scaling ---
scaler = StandardScaler()
X_supervised_scaled = scaler.fit_transform(X_supervised)
X_unsupervised_scaled = scaler.transform(X_unsupervised.reindex(columns=X_supervised.columns, fill_value=0))

# --- Reshape for LSTM Autoencoder (1 timestep) ---
timesteps = 1
input_dim = X_supervised_scaled.shape[1]

X_supervised_lstm = X_supervised_scaled.reshape(-1, timesteps, input_dim)
X_unsupervised_lstm = X_unsupervised_scaled.reshape(-1, timesteps, input_dim)

print(f"Shapes: Supervised {X_supervised_lstm.shape}, Unsupervised {X_unsupervised_lstm.shape}")


MemoryError: Unable to allocate 672. MiB for an array with shape (69, 1276549) and data type float64

In [ ]:
# --- Define Hybrid Autoencoder Model ---
def build_hybrid_autoencoder(timesteps, input_dim, learning_rate=0.0001):
    inputs = Input(shape=(timesteps, input_dim))
    
    # LSTM encoder
    lstm_enc = LSTM(64, activation='relu', return_sequences=True)(inputs)
    lstm_enc = Dropout(0.2)(lstm_enc)
    lstm_enc = LSTM(32, activation='relu', return_sequences=False)(lstm_enc)
    lstm_enc = Dropout(0.2)(lstm_enc)
    
    # FNN encoder
    fnn_input = Flatten()(inputs)
    fnn_enc = Dense(64, activation='relu')(fnn_input)
    fnn_enc = Dropout(0.2)(fnn_enc)
    fnn_enc = Dense(32, activation='relu')(fnn_enc)
    
    # Concatenate encoded features
    concat = Concatenate()([lstm_enc, fnn_enc])
    
    # Decoder
    decoder = Dense(64, activation='relu')(concat)
    decoder = Dropout(0.2)(decoder)
    decoder_output = Dense(timesteps * input_dim, activation='linear')(decoder)
    decoder_reshaped = tf.keras.layers.Reshape((timesteps, input_dim))(decoder_output)
    
    model = Model(inputs, decoder_reshaped)
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse')
    return model

In [ ]:
# --- Train Autoencoder ---
BATCH_SIZE = 256
EPOCHS = 20
LEARNING_RATE = 0.0001

autoencoder = build_hybrid_autoencoder(timesteps, input_dim, LEARNING_RATE)
history = autoencoder.fit(
    X_unsupervised_lstm, X_unsupervised_lstm,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    shuffle=True
)

In [ ]:

# --- Plot Loss ---
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Hybrid Autoencoder Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# --- Calculate Reconstruction Errors on Supervised Data ---
X_supervised_reconstructed = autoencoder.predict(X_supervised_lstm)
supervised_errors = np.mean(np.square(X_supervised_lstm - X_supervised_reconstructed), axis=(1,2))

# --- Plot Reconstruction Error Distribution ---
plt.figure(figsize=(8,5))
sns.histplot(supervised_errors[y_supervised == 0], color='green', bins=50, alpha=0.6, label='Benign', kde=True)
sns.histplot(supervised_errors[y_supervised == 1], color='red', bins=50, alpha=0.6, label='Attack', kde=True)
plt.title('Reconstruction Error by Class')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Count')
plt.legend()
plt.show()


In [ ]:
# Remove NaNs from supervised_errors and corresponding labels
mask = ~np.isnan(supervised_errors)
cleaned_errors = supervised_errors[mask]
cleaned_labels = y_supervised[mask]

auc = roc_auc_score(cleaned_labels, cleaned_errors)
print(f"ROC-AUC Score based on reconstruction error: {auc:.4f}")

fpr, tpr, thresholds = roc_curve(cleaned_labels, cleaned_errors)


In [ ]:
print(np.isnan(X_supervised_scaled).sum())  # Should be 0
print(np.isnan(X_unsupervised_scaled).sum()) # Should be 0


In [ ]:
# --- ROC-AUC Evaluation ---
auc = roc_auc_score(y_supervised, supervised_errors)
print(f"ROC-AUC Score based on reconstruction error: {auc:.4f}")

fpr, tpr, thresholds = roc_curve(y_supervised, supervised_errors)
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Anomaly Detection')
plt.legend()
plt.grid()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

# Using existing fpr, tpr, thresholds from ROC curve
# Find optimal threshold (Youden's J statistic)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal Threshold for anomaly detection: {optimal_threshold:.6f}")

# Plot ROC curve with threshold
plt.figure(figsize=(7,7))
plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
plt.scatter(fpr[optimal_idx], tpr[optimal_idx], marker='o', color='red', label='Optimal Threshold')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Anomaly Detection')
plt.grid(True)
plt.legend()
plt.show()

# Calculate Precision-Recall
precision, recall, pr_thresholds = precision_recall_curve(cleaned_labels, cleaned_errors)
ap_score = average_precision_score(cleaned_labels, cleaned_errors)
print(f'Average Precision Score: {ap_score:.4f}')

# Plot Precision-Recall Curve
plt.figure(figsize=(7,7))
plt.plot(recall, precision, label=f'AP = {ap_score:.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for Anomaly Detection')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from cvss import CVSS3
import pandas as pd
print("import completed")

# 1. Train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_supervised_scaled, y_supervised)

print("done with train classifier")
# 2. Predict attack labels on supervised data
y_pred = clf.predict(X_supervised_scaled)

print("Attack Classification Report:")
print(classification_report(y_supervised, y_pred, target_names=['Benign', 'Attack']))
print(f"Classification Accuracy: {accuracy_score(y_supervised, y_pred):.4f}")

In [ ]:
import numpy as np
import pandas as pd

# --- CVSS Lookup Table (example, fill in your real scores/classes) ---
cvss_lookup = {
    'DOS_HULK': 7.5,
    'DDOS': 8.0,
    'FTP-PATATOR': 6.0,
    'SSH-PATATOR': 6.5,
    'PORTSCAN': 5.0,
    'BENIGN': 0.0,
}

# 1 Predict with Random Forest for known types 
rf_pred_labels = clf.predict(X_supervised_scaled)

# 2: Calculate autoencoder reconstruction errors 
X_supervised_lstm = X_supervised_scaled.reshape(-1, 1, X_supervised_scaled.shape[1])
X_supervised_recon = autoencoder.predict(X_supervised_lstm)
recon_errors = np.mean(np.square(X_supervised_lstm - X_supervised_recon), axis=(1,2))

# 3: Hybrid Decision Logic
threshold = 0.076033  
final_labels = []
for rf_label, err in zip(rf_pred_labels, recon_errors):
    if err <= threshold:
        final_labels.append(rf_label)    
    else:
        final_labels.append('Unknown')   # Unknown anomaly (dynamic detection)

# 4: Assign CVSS Scores 
cvss_scores = []
for label in final_labels:
    cvss_scores.append(cvss_lookup.get(label, 'Dynamic/Unknown'))

hybrid_df = pd.DataFrame({
    'RF_Label': rf_pred_labels,
    'Reconstruction_Error': recon_errors,
    'Hybrid_Label': final_labels,
    'CVSS_Score': cvss_scores
})

print(hybrid_df['Hybrid_Label'].value_counts())
print(hybrid_df.groupby('Hybrid_Label')['CVSS_Score'].value_counts())

# --- Optional: For new or unsupervised data ---
X_unsup_lstm = X_unsupervised_scaled.reshape(-1, 1, X_unsupervised_scaled.shape[1])
X_unsup_recon = autoencoder.predict(X_unsup_lstm)
unsup_errors = np.mean(np.square(X_unsup_lstm - X_unsup_recon), axis=(1,2))
unsup_labels = ['Unknown' if e > threshold else 'Benign' for e in unsup_errors]
unsup_cvss = [cvss_lookup.get(l, 'Dynamic/Unknown') for l in unsup_labels]
unsup_df = pd.DataFrame({'Unsupervised_Label': unsup_labels, 'Error': unsup_errors, 'CVSS_Score': unsup_cvss})

# --- Save or further process results ---
hybrid_df.to_csv('hybrid_supervised_results.csv', index=False)
unsup_df.to_csv('hybrid_unsupervised_results.csv', index=False)
